# Build BOW, DTM, TFIDF, and Reduced TFIDF_L2

This notebook adapts the MO7 homework workflow to the constitution corpus. It builds a bag-of-words table, a document-term matrix, a TFIDF matrix, and a reduced + L2-normalized TFIDF matrix. The main stages are also exposed as reusable functions: `generate_bow()`, `build_dtm()`, `build_tfidf()`, and `build_reduced_tfidf_l2()`.

## Inputs and Outputs

**Inputs**
- `corpus.csv`
- `vocab.csv`

**Outputs**
- `bow.csv`
- `dtm.csv`
- `tfidf.csv`
- `tfidf_reduced_l2.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfTransformer

CORPUS_CSV = Path('corpus.csv')
VOCAB_CSV = Path('vocab.csv')

BOW_CSV = Path('bow.csv')
DTM_CSV = Path('dtm.csv')
TFIDF_CSV = Path('tfidf.csv')
TFIDF_REDUCED_L2_CSV = Path('tfidf_reduced_l2.csv')

# Choose the bag level for documents in the matrix.
BAG_LEVELS = {
    'DOCS': ['country'],
    'ARTICLES': ['country', 'article_n'],
    'CLAUSES': ['country', 'article_n', 'clause_n'],
}
BAG_NAME = 'DOCS'

# Reduction settings adapted from the MO7 homework pattern.
MIN_DF = 50
MIN_I = 10


## Load CORPUS and VOCAB

In [2]:
# Load the normalized corpus and vocabulary inputs for table building.
CORPUS = pd.read_csv(
    CORPUS_CSV,
    usecols=['country', 'article_n', 'clause_n', 'term_str']
)
# Clean empty terms so the matrix steps only see valid vocabulary items.
CORPUS['term_str'] = CORPUS['term_str'].fillna('').astype(str).str.strip()
CORPUS = CORPUS[CORPUS['term_str'] != ''].copy()

# Align term metadata by term string for later filtering.
VOCAB = pd.read_csv(VOCAB_CSV).set_index('term_str')

print(f'Corpus rows: {len(CORPUS):,}')
print(f'Unique terms: {CORPUS["term_str"].nunique():,}')
print(f'Bag level: {BAG_NAME} -> {BAG_LEVELS[BAG_NAME]}')
CORPUS.head()


Corpus rows: 3,848,633
Unique terms: 24,769
Bag level: DOCS -> ['country']


,country,article_n,clause_n,term_str
0,Afghanistan,1,1,in
1,Afghanistan,1,1,the
2,Afghanistan,1,1,name
3,Afghanistan,1,1,of
4,Afghanistan,1,1,allah


## Build BOW and DTM

This stage is organized into two callable steps: `generate_bow()` builds the long bag-of-words table, and `build_dtm()` converts that table into the wide document-term matrix.

In [3]:
def generate_bow(tokens: pd.DataFrame, bag_name: str) -> pd.DataFrame:
    # Group tokens at the selected bag level and count term occurrences.
    bag_cols = BAG_LEVELS[bag_name]
    bow = (
        tokens.groupby(bag_cols + ['term_str'])['term_str']
        .count()
        .to_frame('n')
        .sort_index()
    )
    return bow


def build_dtm(bow: pd.DataFrame) -> pd.DataFrame:
    # Unstack the long BOW table into a bag-by-term matrix.
    return bow['n'].unstack(fill_value=0).sort_index(axis=0).sort_index(axis=1)


BOW = generate_bow(CORPUS, BAG_NAME)
DTM = build_dtm(BOW)

print('BOW shape:', BOW.shape)
print('DTM shape:', DTM.shape)
BOW.head()


BOW shape: (356245, 1)
DTM shape: (192, 24769)


n
country     term_str     
Afghanistan ability     1
            about       7
            abrogation  3
            absence     4
            academic    2

## Build TFIDF

This uses `sklearn.feature_extraction.text.TfidfTransformer(norm=None, smooth_idf=True)`, so for term `t` in document bag `d`, the notebook computes `tfidf(t, d) = tf(t, d) * idf(t)`, where `tf(t, d)` is the raw count from `DTM` and `idf(t) = log((1 + N) / (1 + df(t))) + 1`. Here `N` is the number of document bags and `df(t)` is the number of bags containing term `t`. Because `norm=None`, the full `TFIDF` table keeps these unnormalized weights; L2 normalization is applied later only after feature reduction when `TFIDF_REDUCED_L2` is built.

In [4]:
def build_tfidf(dtm: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    # Compute document frequency alongside the unnormalized TFIDF matrix.
    doc_freq = (dtm > 0).sum(axis=0)
    tfidf_transformer = TfidfTransformer(norm=None, smooth_idf=True)
    tfidf = tfidf_transformer.fit_transform(dtm)
    tfidf = pd.DataFrame(
        tfidf.toarray(),
        index=dtm.index,
        columns=dtm.columns,
    )
    return tfidf, doc_freq


TFIDF, doc_freq = build_tfidf(DTM)

print('TFIDF shape:', TFIDF.shape)
TFIDF.iloc[:5, :5]


TFIDF shape: (192, 24769)


term_str,aa,aalim,aargau,ab,aba
country,,,,,
Afghanistan,0.0,0.0,0.0,0.0,0.0
Albania,0.0,0.0,0.0,0.0,0.0
Algeria,0.0,0.0,0.0,0.0,0.0
Andorra,0.0,0.0,0.0,0.0,0.0
Angola,0.0,0.0,0.0,0.0,0.0


## Reduce Features and Build TFIDF_L2

The reduction rule follows the MO7 homework idea: keep terms that appear in at least `MIN_DF` bags and have self-information `i >= MIN_I` in `VOCAB`. After that vocabulary filter, each reduced TFIDF row is L2-normalized by dividing by the square root of the sum of squared TFIDF weights in that row.

## Sample TFIDF Columns

This shows a reproducible random sample of 20 term columns from the full TFIDF matrix for quick inspection in the notebook.

In [5]:
TFIDF_SAMPLE = TFIDF.sample(n=20, axis=1, random_state=42)
print('Sampled TFIDF columns:')
print(', '.join(TFIDF_SAMPLE.columns.tolist()))
TFIDF_SAMPLE.head()

Sampled TFIDF columns:
inclinations, isuikwato, cultivation, prejudicing, non-believers, usefully, gem, receivable, jordan, technical-and, functional, disruption, uri, dar, chhattisgarh, regionalism, enactmentsthe, changeable, deficiency, sapele


term_str,inclinations,isuikwato,cultivation,prejudicing,non-believers,usefully,gem,receivable,jordan,technical-and,functional,disruption,uri,dar,chhattisgarh,regionalism,enactmentsthe,changeable,deficiency,sapele
country,,,,,,,,,,,,,,,,,,,,
Afghanistan,0.0,0.0,3.623633,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Albania,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Algeria,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Andorra,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Angola,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
def build_reduced_tfidf_l2(
    dtm: pd.DataFrame,
    tfidf: pd.DataFrame,
    vocab: pd.DataFrame,
    doc_freq: pd.Series,
    min_df: int,
    min_i: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    # Pull the information score onto the DTM vocabulary order before filtering.
    term_info = vocab.reindex(dtm.columns)['i'].fillna(0)
    mask = (doc_freq >= min_df) & (term_info >= min_i)

    # Filter the count and TFIDF matrices with the same vocabulary mask.
    dtm_reduced = dtm.loc[:, mask]
    tfidf_reduced = tfidf.loc[:, mask]

    # Normalize each document vector so later models compare direction over length.
    row_norms = np.sqrt((tfidf_reduced ** 2).sum(axis=1)).replace(0, 1)
    tfidf_reduced_l2 = tfidf_reduced.div(row_norms, axis=0)
    return dtm_reduced, tfidf_reduced_l2, term_info


DTM_REDUCED, TFIDF_REDUCED_L2, term_info = build_reduced_tfidf_l2(
    DTM,
    TFIDF,
    VOCAB,
    doc_freq,
    MIN_DF,
    MIN_I,
)

print('Reduced DTM shape:', DTM_REDUCED.shape)
print('Reduced TFIDF_L2 shape:', TFIDF_REDUCED_L2.shape)
TFIDF_REDUCED_L2.iloc[:5, :5]


Reduced DTM shape: (192, 2125)
Reduced TFIDF_L2 shape: (192, 2125)


term_str,abide,ability,able,abolish,abolished
country,,,,,
Afghanistan,0.000000,0.008401,0.000000,0.0,0.0
Albania,0.000000,0.000000,0.012931,0.0,0.0
Algeria,0.000000,0.000000,0.000000,0.0,0.0
Andorra,0.000000,0.000000,0.000000,0.0,0.0
Angola,0.009864,0.004208,0.003558,0.0,0.0


## Save Tables

In [7]:
# Save each derived table so later modeling notebooks can load them directly.
BOW.reset_index().to_csv(BOW_CSV, index=False)
DTM.to_csv(DTM_CSV)
TFIDF.to_csv(TFIDF_CSV)
TFIDF_REDUCED_L2.to_csv(TFIDF_REDUCED_L2_CSV)

print(f'Saved BOW to: {BOW_CSV.resolve()}')
print(f'Saved DTM to: {DTM_CSV.resolve()}')
print(f'Saved TFIDF to: {TFIDF_CSV.resolve()}')
print(f'Saved reduced TFIDF_L2 to: {TFIDF_REDUCED_L2_CSV.resolve()}')

Saved BOW to: C:\Users\garre\school\spring_2026\ds_5001\bow.csv
Saved DTM to: C:\Users\garre\school\spring_2026\ds_5001\dtm.csv
Saved TFIDF to: C:\Users\garre\school\spring_2026\ds_5001\tfidf.csv
Saved reduced TFIDF_L2 to: C:\Users\garre\school\spring_2026\ds_5001\tfidf_reduced_l2.csv
